# ISIC 2019 EfficientNet-B0 Feature-Space Analysis

**Dataset:** ISIC 2019  
**Task:** Binary skin lesion classification and latent feature analysis  
**Model:** EfficientNet-B0  
**Methods:** PCA, t-SNE, clustering metrics, Grad-CAM, Grad-CAM++, GradientSHAP and Integrated Gradients

This notebook studies both predictive performance and the organisation of learned representations in feature space. It was developed in Google Colab; dataset, checkpoint and output paths must be adjusted before execution.

In [ ]:


!pip install -q timm

import os
import glob
import copy
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    davies_bouldin_score,
    calinski_harabasz_score,
    silhouette_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE



if os.path.exists("/content/drive/MyDrive"):
    print("Google Drive already mounted.")
else:
    from google.colab import drive
    drive.mount("/content/drive")



SEED = 42

MODEL_NAME = "efficientnet_b0"
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0

EPOCHS = 12
LR = 2e-4
WEIGHT_DECAY = 1e-4
USE_AMP = True

MAX_PER_BINARY_CLASS = 3000
MAX_TSNE_PER_CLASS = None

STAGE_EPOCHS = [1, 5, 10]

OUTPUT_DIR = "/content/drive/MyDrive/ISIC2019_EFFICIENTNETB0_SPATIAL_EXPERIMENT"
LOCAL_DATA_DIR = "/content/isic2019_spatial_local"

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
SPATIAL_CKPT_DIR = os.path.join(OUTPUT_DIR, "spatial_stage_checkpoints")
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
SPATIAL_DIR = os.path.join(OUTPUT_DIR, "spatial_feature_space_analysis")
SPATIAL_PLOTS_DIR = os.path.join(SPATIAL_DIR, "plots")
SPATIAL_TABLES_DIR = os.path.join(SPATIAL_DIR, "tables")

for d in [
    OUTPUT_DIR,
    LOCAL_DATA_DIR,
    CHECKPOINT_DIR,
    SPATIAL_CKPT_DIR,
    RESULTS_DIR,
    SPATIAL_DIR,
    SPATIAL_PLOTS_DIR,
    SPATIAL_TABLES_DIR
]:
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("OUTPUT_DIR:", OUTPUT_DIR)



def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)



required_folders = {"AK", "BCC", "BKL", "DF", "MEL", "NV", "SCC", "VASC"}

def find_isic2019_root(start_dir="/content/drive/MyDrive"):
    candidates = []

    for root, dirs, files in os.walk(start_dir):
        if required_folders.issubset(set(dirs)):
            candidates.append(root)

    if len(candidates) == 0:
        raise FileNotFoundError(
            "Δεν βρέθηκε φάκελος με AK, BCC, BKL, DF, MEL, NV, SCC, VASC."
        )

    candidates = sorted(candidates, key=lambda x: len(x))

    print("Candidate ISIC 2019 folders:")
    for i, c in enumerate(candidates[:10]):
        print(i, ":", c)

    return candidates[0]

ISIC_ROOT = find_isic2019_root("/content/drive/MyDrive")

print("\nUsing ISIC_ROOT:", ISIC_ROOT)
print("Contents:", os.listdir(ISIC_ROOT))



benign_classes = ["NV", "BKL", "DF", "VASC"]
suspicious_classes = ["MEL", "BCC", "AK", "SCC"]

binary_map = {}
for c in benign_classes:
    binary_map[c] = 0
for c in suspicious_classes:
    binary_map[c] = 1

binary_name_map = {
    0: "benign_like",
    1: "suspicious"
}

rows = []

for class_name in sorted(required_folders):
    class_dir = os.path.join(ISIC_ROOT, class_name)

    image_paths = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        image_paths.extend(glob.glob(os.path.join(class_dir, ext)))

    print(class_name, "images:", len(image_paths))

    for p in image_paths:
        rows.append({
            "image_path_original": p,
            "class_name": class_name,
            "binary_label": binary_map[class_name],
            "binary_class": binary_name_map[binary_map[class_name]]
        })

df_all = pd.DataFrame(rows)

print("\nOriginal multiclass distribution:")
print(df_all["class_name"].value_counts())

print("\nBinary distribution before balancing:")
print(df_all["binary_class"].value_counts())

df_benign = df_all[df_all["binary_label"] == 0].copy()
df_suspicious = df_all[df_all["binary_label"] == 1].copy()

n_benign = min(MAX_PER_BINARY_CLASS, len(df_benign))
n_suspicious = min(MAX_PER_BINARY_CLASS, len(df_suspicious))

df_benign = df_benign.sample(n=n_benign, random_state=SEED)
df_suspicious = df_suspicious.sample(n=n_suspicious, random_state=SEED)

df = pd.concat([df_benign, df_suspicious], axis=0)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("\nBalanced binary distribution:")
print(df["binary_class"].value_counts())
print("Total:", len(df))

df_path = os.path.join(OUTPUT_DIR, "isic2019_binary_dataframe.csv")
df.to_csv(df_path, index=False)
print("Saved dataframe:", df_path)



print("\nCopying images locally...")

local_paths = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    src = row["image_path_original"]
    cls = row["binary_class"]
    fname = os.path.basename(src)

    dst_dir = os.path.join(LOCAL_DATA_DIR, cls)
    os.makedirs(dst_dir, exist_ok=True)

    dst = os.path.join(dst_dir, fname)

    if not os.path.exists(dst):
        shutil.copy2(src, dst)

    local_paths.append(dst)

df["image_path"] = local_paths

print("Local copy completed.")



train_val_df, test_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["binary_label"],
    random_state=SEED
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.1765,
    stratify=train_val_df["binary_label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nSplit sizes:")
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

print("\nTrain distribution:")
print(train_df["binary_class"].value_counts())

print("\nVal distribution:")
print(val_df["binary_class"].value_counts())

print("\nTest distribution:")
print(test_df["binary_class"].value_counts())

train_df.to_csv(os.path.join(OUTPUT_DIR, "train_split.csv"), index=False)
val_df.to_csv(os.path.join(OUTPUT_DIR, "val_split.csv"), index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, "test_split.csv"), index=False)


def safe_open_rgb(path):
    try:
        img = cv2.imread(path, cv2.IMREAD_COLOR)

        if img is None:
            raise ValueError("cv2.imread returned None")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return Image.fromarray(img)

    except Exception as e:
        print("Image error:", path, e)
        arr = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        return Image.fromarray(arr)

train_tf = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.82, 1.0), ratio=(0.90, 1.10)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.2),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.10, contrast=0.12, saturation=0.08, hue=0.02),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

class ISICBinaryDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = safe_open_rgb(row["image_path"])

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(float(row["binary_label"]), dtype=torch.float32)

        return image, label

train_ds = ISICBinaryDataset(train_df, transform=train_tf)
val_ds = ISICBinaryDataset(val_df, transform=eval_tf)
test_ds = ISICBinaryDataset(test_df, transform=eval_tf)

train_labels = train_df["binary_label"].values.astype(int)
class_counts = np.bincount(train_labels, minlength=2)
class_counts = np.maximum(class_counts, 1)
sample_weights = (1.0 / class_counts)[train_labels]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)



def create_model(pretrained=True):
    model = timm.create_model(
        MODEL_NAME,
        pretrained=pretrained,
        num_classes=1
    )
    return model

model = create_model(pretrained=True).to(device)

pos = int(train_df["binary_label"].sum())
neg = int((train_df["binary_label"] == 0).sum())

pos_weight = torch.tensor([neg / max(pos, 1)], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = torch.cuda.amp.GradScaler(
    enabled=(USE_AMP and device.type == "cuda")
)


initial_ckpt_path = os.path.join(SPATIAL_CKPT_DIR, "stage_before_training.pth")

torch.save({
    "stage": "before_training",
    "epoch": 0,
    "model_name": MODEL_NAME,
    "model_state": copy.deepcopy(model.state_dict()),
    "binary_name_map": binary_name_map,
    "img_size": IMG_SIZE
}, initial_ckpt_path)

print("Saved initial checkpoint:", initial_ckpt_path)



def evaluate_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob),
        "pred": y_pred
    }

def find_best_threshold(y_true, y_prob, metric="f1"):
    thresholds = np.linspace(0.05, 0.95, 91)

    best_threshold = 0.5
    best_score = -1

    for th in thresholds:
        m = evaluate_at_threshold(y_true, y_prob, th)

        if m[metric] > best_score:
            best_score = m[metric]
            best_threshold = th

    return best_threshold, best_score

@torch.no_grad()
def get_probs_labels(model, loader):
    model.eval()

    probs_all = []
    labels_all = []

    for images, labels in tqdm(loader, desc="Predicting", leave=False):
        images = images.to(device)

        logits = model(images).view(-1)
        probs = torch.sigmoid(logits).detach().cpu().numpy()

        probs_all.extend(probs)
        labels_all.extend(labels.numpy())

    return np.array(labels_all).astype(int), np.array(probs_all)



best_val_auc = -1
best_state = None
best_epoch = -1
best_threshold = 0.5

history = []

for epoch in range(1, EPOCHS + 1):
    model.train()

    running_loss = 0.0
    n = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
            logits = model(images).view(-1)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        n += images.size(0)

        loop.set_postfix(loss=f"{loss.item():.4f}")

    scheduler.step()

    train_loss = running_loss / max(n, 1)

    y_val, p_val = get_probs_labels(model, val_loader)
    threshold, _ = find_best_threshold(y_val, p_val, metric="f1")
    val_metrics = evaluate_at_threshold(y_val, p_val, threshold)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_acc={val_metrics['accuracy']:.4f} | "
        f"val_precision={val_metrics['precision']:.4f} | "
        f"val_recall={val_metrics['recall']:.4f} | "
        f"val_f1={val_metrics['f1']:.4f} | "
        f"val_auc={val_metrics['auc']:.4f} | "
        f"thr={threshold:.2f}"
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "val_auc": val_metrics["auc"],
        "threshold": threshold
    })


    epoch_ckpt_path = os.path.join(CHECKPOINT_DIR, f"epoch_{epoch}.pth")

    torch.save({
        "epoch": epoch,
        "model_state": copy.deepcopy(model.state_dict()),
        "threshold": threshold,
        "val_auc": val_metrics["auc"],
        "val_f1": val_metrics["f1"],
        "model_name": MODEL_NAME,
        "img_size": IMG_SIZE
    }, epoch_ckpt_path)


    if epoch in STAGE_EPOCHS:
        stage_ckpt_path = os.path.join(SPATIAL_CKPT_DIR, f"stage_epoch_{epoch}.pth")

        torch.save({
            "stage": f"epoch_{epoch}",
            "epoch": epoch,
            "model_name": MODEL_NAME,
            "model_state": copy.deepcopy(model.state_dict()),
            "threshold": threshold,
            "val_auc": val_metrics["auc"],
            "val_f1": val_metrics["f1"],
            "binary_name_map": binary_name_map,
            "img_size": IMG_SIZE
        }, stage_ckpt_path)

        print("Saved spatial checkpoint:", stage_ckpt_path)


    if not np.isnan(val_metrics["auc"]) and val_metrics["auc"] > best_val_auc:
        best_val_auc = val_metrics["auc"]
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        best_threshold = threshold

        best_ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pth")
        best_stage_ckpt_path = os.path.join(SPATIAL_CKPT_DIR, "stage_best_model.pth")

        torch.save({
            "epoch": best_epoch,
            "model_state": best_state,
            "threshold": best_threshold,
            "best_val_auc": best_val_auc,
            "model_name": MODEL_NAME,
            "img_size": IMG_SIZE
        }, best_ckpt_path)

        torch.save({
            "stage": "best_model",
            "epoch": best_epoch,
            "model_name": MODEL_NAME,
            "model_state": best_state,
            "threshold": best_threshold,
            "best_val_auc": best_val_auc,
            "binary_name_map": binary_name_map,
            "img_size": IMG_SIZE
        }, best_stage_ckpt_path)

        print("Saved best model:", best_ckpt_path)

history_df = pd.DataFrame(history)
history_path = os.path.join(RESULTS_DIR, "training_history.csv")
history_df.to_csv(history_path, index=False)

display(history_df)

model.load_state_dict(best_state)
model.eval()



y_test, p_test = get_probs_labels(model, test_loader)

test_metrics = evaluate_at_threshold(y_test, p_test, best_threshold)

print("\n=== TEST RESULTS ===")
print("Best epoch:", best_epoch)
print("Threshold:", best_threshold)
print("Accuracy:", test_metrics["accuracy"])
print("Precision:", test_metrics["precision"])
print("Recall:", test_metrics["recall"])
print("F1:", test_metrics["f1"])
print("AUC:", test_metrics["auc"])

cm = confusion_matrix(y_test, test_metrics["pred"])
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_metrics["pred"],
        target_names=["benign_like", "suspicious"],
        zero_division=0
    )
)

test_summary_df = pd.DataFrame([{
    "model": MODEL_NAME,
    "dataset": "ISIC 2019",
    "task": "binary suspicious vs benign-like",
    "best_epoch": best_epoch,
    "threshold": best_threshold,
    "accuracy": test_metrics["accuracy"],
    "precision": test_metrics["precision"],
    "recall": test_metrics["recall"],
    "f1": test_metrics["f1"],
    "auc": test_metrics["auc"]
}])

test_summary_path = os.path.join(RESULTS_DIR, "test_summary.csv")
test_summary_df.to_csv(test_summary_path, index=False)

display(test_summary_df)

test_predictions_df = test_df.copy().reset_index(drop=True)
test_predictions_df["true_label"] = y_test
test_predictions_df["prob_suspicious"] = p_test
test_predictions_df["pred_label"] = test_metrics["pred"]
test_predictions_df["true_class"] = test_predictions_df["true_label"].map(binary_name_map)
test_predictions_df["pred_class"] = test_predictions_df["pred_label"].map(binary_name_map)
test_predictions_df["correct"] = test_predictions_df["true_label"] == test_predictions_df["pred_label"]

test_predictions_path = os.path.join(RESULTS_DIR, "test_predictions.csv")
test_predictions_df.to_csv(test_predictions_path, index=False)

plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix - ISIC 2019 EfficientNet-B0")
plt.colorbar()

tick_marks = np.arange(2)
plt.xticks(tick_marks, ["benign_like", "suspicious"])
plt.yticks(tick_marks, ["benign_like", "suspicious"])

plt.xlabel("Predicted")
plt.ylabel("True")

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.tight_layout()

cm_path = os.path.join(RESULTS_DIR, "confusion_matrix.png")
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()



stage_ckpts = {
    "before_training": os.path.join(SPATIAL_CKPT_DIR, "stage_before_training.pth"),
    "epoch_1": os.path.join(SPATIAL_CKPT_DIR, "stage_epoch_1.pth"),
    "epoch_5": os.path.join(SPATIAL_CKPT_DIR, "stage_epoch_5.pth"),
    "epoch_10": os.path.join(SPATIAL_CKPT_DIR, "stage_epoch_10.pth"),
    "best_model": os.path.join(SPATIAL_CKPT_DIR, "stage_best_model.pth")
}

def load_stage_model(ckpt_path):
    # Safe only for checkpoints created by this notebook; never load untrusted files.
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    m = create_model(pretrained=False).to(device)
    m.load_state_dict(ckpt["model_state"])
    m.eval()

    return m, ckpt

@torch.no_grad()
def extract_features_for_spatial(m, loader):
    m.eval()

    feats = []
    labs = []
    probs = []

    for x, y in tqdm(loader, desc="Extracting features", leave=False):
        x = x.to(device)

        z = m.forward_features(x)

        try:
            z = m.forward_head(z, pre_logits=True)
        except Exception:
            if z.ndim == 4:
                z = z.mean(dim=(2, 3))
            elif z.ndim == 3:
                z = z.mean(dim=1)

        if z.ndim == 4:
            z = z.mean(dim=(2, 3))
        elif z.ndim == 3:
            z = z.mean(dim=1)

        logits = m(x).view(-1)
        p = torch.sigmoid(logits)

        feats.append(z.detach().cpu().numpy())
        labs.append(y.numpy().astype(int))
        probs.append(p.detach().cpu().numpy())

    feats = np.concatenate(feats)
    labs = np.concatenate(labs)
    probs = np.concatenate(probs)

    return feats, labs, probs

def make_balanced_indices(labels, max_per_class=None, seed=42):
    rng = np.random.default_rng(seed)
    labels = np.array(labels)
    classes = np.unique(labels)

    n_per_class = min([np.sum(labels == c) for c in classes])

    if max_per_class is not None:
        n_per_class = min(n_per_class, max_per_class)

    selected = []

    for c in classes:
        idx = np.where(labels == c)[0]
        chosen = rng.choice(idx, size=n_per_class, replace=False)
        selected.extend(chosen.tolist())

    selected = np.array(selected)
    rng.shuffle(selected)

    return selected

def compute_spatial_metrics(features_2d, labels):
    labels = np.array(labels)

    db = davies_bouldin_score(features_2d, labels)
    ch = calinski_harabasz_score(features_2d, labels)
    sil = silhouette_score(features_2d, labels)

    class0 = features_2d[labels == 0]
    class1 = features_2d[labels == 1]

    centroid0 = class0.mean(axis=0)
    centroid1 = class1.mean(axis=0)

    centroid_distance = np.linalg.norm(centroid1 - centroid0)

    spread0 = np.mean(np.linalg.norm(class0 - centroid0, axis=1))
    spread1 = np.mean(np.linalg.norm(class1 - centroid1, axis=1))

    mean_intra_class_spread = (spread0 + spread1) / 2
    separation_ratio = centroid_distance / (mean_intra_class_spread + 1e-8)

    return {
        "davies_bouldin": db,
        "calinski_harabasz": ch,
        "silhouette": sil,
        "centroid_distance": centroid_distance,
        "benign_like_spread": spread0,
        "suspicious_spread": spread1,
        "mean_intra_class_spread": mean_intra_class_spread,
        "separation_ratio": separation_ratio
    }

tmp_model, _ = load_stage_model(stage_ckpts["best_model"])
tmp_features, tmp_labels, tmp_probs = extract_features_for_spatial(tmp_model, test_loader)

balanced_indices = make_balanced_indices(
    tmp_labels,
    max_per_class=MAX_TSNE_PER_CLASS,
    seed=SEED
)

del tmp_model
torch.cuda.empty_cache()

print("\nBalanced spatial subset:")
print("Total:", len(balanced_indices))
print("benign_like:", int(np.sum(tmp_labels[balanced_indices] == 0)))
print("suspicious:", int(np.sum(tmp_labels[balanced_indices] == 1)))

stage_order = ["before_training", "epoch_1", "epoch_5", "epoch_10", "best_model"]

stage_display_names = {
    "before_training": "Before training",
    "epoch_1": "Epoch 1",
    "epoch_5": "Epoch 5",
    "epoch_10": "Epoch 10",
    "best_model": "Best model"
}

stage_results = []
stage_coordinates = {}

for stage_name in stage_order:
    print("\n" + "=" * 90)
    print("Spatial analysis stage:", stage_name)
    print("=" * 90)

    stage_model, stage_ckpt = load_stage_model(stage_ckpts[stage_name])

    features, labels, probs = extract_features_for_spatial(stage_model, test_loader)

    features_used = features[balanced_indices]
    labels_used = labels[balanced_indices]
    probs_used = probs[balanced_indices]

    scaler_features = StandardScaler()
    features_scaled = scaler_features.fit_transform(features_used)

    n_components = min(50, features_scaled.shape[1], features_scaled.shape[0] - 1)

    pca = PCA(n_components=n_components, random_state=SEED)
    features_pca = pca.fit_transform(features_scaled)
    pca_var = pca.explained_variance_ratio_.sum()

    perplexity = min(30, max(5, (features_pca.shape[0] - 1) // 3))

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        learning_rate="auto",
        init="pca",
        max_iter=1500,
        random_state=SEED
    )

    features_2d = tsne.fit_transform(features_pca)

    metrics = compute_spatial_metrics(features_2d, labels_used)

    row = {
        "stage": stage_name,
        "stage_display": stage_display_names[stage_name],
        "epoch": stage_ckpt.get("epoch", 0),
        "num_samples": len(labels_used),
        "num_benign_like": int(np.sum(labels_used == 0)),
        "num_suspicious": int(np.sum(labels_used == 1)),
        "pca_components": n_components,
        "pca_explained_variance": pca_var,
        "tsne_perplexity": perplexity,
        **metrics
    }

    stage_results.append(row)

    coords_df = pd.DataFrame({
        "stage": stage_name,
        "stage_display": stage_display_names[stage_name],
        "tsne_1": features_2d[:, 0],
        "tsne_2": features_2d[:, 1],
        "label": labels_used,
        "class_name": [binary_name_map[int(x)] for x in labels_used],
        "prob_suspicious": probs_used
    })

    stage_coordinates[stage_name] = coords_df

    coord_path = os.path.join(SPATIAL_TABLES_DIR, f"coordinates_{stage_name}.csv")
    coords_df.to_csv(coord_path, index=False)

    print("Metrics:")
    print(row)

    del stage_model
    torch.cuda.empty_cache()

spatial_summary_df = pd.DataFrame(stage_results)

spatial_summary_path = os.path.join(
    SPATIAL_TABLES_DIR,
    "spatial_stage_comparison_table.csv"
)

spatial_summary_df.to_csv(spatial_summary_path, index=False)

print("\n=== SPATIAL STAGE COMPARISON TABLE ===")
display(spatial_summary_df)

baseline_row = spatial_summary_df[
    spatial_summary_df["stage"] == "before_training"
].iloc[0]

interpret_rows = []

for _, row in spatial_summary_df.iterrows():
    interpret_rows.append({
        "stage": row["stage_display"],
        "davies_bouldin": row["davies_bouldin"],
        "davies_bouldin_change_vs_before": row["davies_bouldin"] - baseline_row["davies_bouldin"],
        "calinski_harabasz": row["calinski_harabasz"],
        "calinski_change_vs_before": row["calinski_harabasz"] - baseline_row["calinski_harabasz"],
        "silhouette": row["silhouette"],
        "silhouette_change_vs_before": row["silhouette"] - baseline_row["silhouette"],
        "centroid_distance": row["centroid_distance"],
        "centroid_distance_change_vs_before": row["centroid_distance"] - baseline_row["centroid_distance"],
        "mean_intra_class_spread": row["mean_intra_class_spread"],
        "separation_ratio": row["separation_ratio"],
        "separation_ratio_change_vs_before": row["separation_ratio"] - baseline_row["separation_ratio"]
    })

interpretation_df = pd.DataFrame(interpret_rows)

interpretation_path = os.path.join(
    SPATIAL_TABLES_DIR,
    "spatial_stage_changes_vs_before_training.csv"
)

interpretation_df.to_csv(interpretation_path, index=False)

print("\n=== WHAT CHANGED COMPARED TO BEFORE TRAINING ===")
display(interpretation_df)

compact_table_df = spatial_summary_df[[
    "stage_display",
    "epoch",
    "davies_bouldin",
    "calinski_harabasz",
    "silhouette",
    "centroid_distance",
    "mean_intra_class_spread",
    "separation_ratio"
]].copy()

compact_table_df = compact_table_df.rename(columns={
    "stage_display": "Stage",
    "epoch": "Epoch",
    "davies_bouldin": "Davies-Bouldin ↓",
    "calinski_harabasz": "Calinski-Harabasz ↑",
    "silhouette": "Silhouette ↑",
    "centroid_distance": "Centroid distance ↑",
    "mean_intra_class_spread": "Mean intra-class spread ↓",
    "separation_ratio": "Separation ratio ↑"
})

compact_table_path = os.path.join(
    SPATIAL_TABLES_DIR,
    "compact_spatial_comparison_table_for_thesis.csv"
)

compact_table_df.to_csv(compact_table_path, index=False)

print("\n=== COMPACT THESIS TABLE ===")
display(compact_table_df)



fig, axes = plt.subplots(1, len(stage_order), figsize=(5 * len(stage_order), 5))

for ax, stage_name in zip(axes, stage_order):
    coords = stage_coordinates[stage_name]

    for label_value, class_name in binary_name_map.items():
        mask = coords["label"].values == label_value

        ax.scatter(
            coords.loc[mask, "tsne_1"],
            coords.loc[mask, "tsne_2"],
            s=24,
            alpha=0.82,
            label=class_name
        )

    row = spatial_summary_df[spatial_summary_df["stage"] == stage_name].iloc[0]

    ax.set_title(
        f"{stage_display_names[stage_name]}\n"
        f"Sil={row['silhouette']:.3f} | DB={row['davies_bouldin']:.3f}"
    )

    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.grid(True)

axes[0].legend()

plt.suptitle(
    "Feature Space Evolution Across Training Stages\n"
    "ISIC 2019 binary suspicious vs benign-like",
    fontsize=16
)

plt.tight_layout()

multi_stage_plot_path = os.path.join(
    SPATIAL_PLOTS_DIR,
    "feature_space_evolution_by_class.png"
)

plt.savefig(multi_stage_plot_path, dpi=300, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, len(stage_order), figsize=(5 * len(stage_order), 5))

for ax, stage_name in zip(axes, stage_order):
    coords = stage_coordinates[stage_name]

    sc = ax.scatter(
        coords["tsne_1"],
        coords["tsne_2"],
        c=coords["prob_suspicious"],
        s=24,
        alpha=0.85,
        vmin=0,
        vmax=1
    )

    row = spatial_summary_df[spatial_summary_df["stage"] == stage_name].iloc[0]

    ax.set_title(
        f"{stage_display_names[stage_name]}\n"
        f"CH={row['calinski_harabasz']:.1f}"
    )

    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.grid(True)

fig.colorbar(sc, ax=axes.ravel().tolist(), shrink=0.75, label="Probability suspicious")

plt.suptitle(
    "Decision Confidence Evolution in Feature Space\n"
    "Color indicates probability for suspicious class",
    fontsize=16
)

prob_space_plot_path = os.path.join(
    SPATIAL_PLOTS_DIR,
    "feature_space_evolution_by_probability.png"
)

plt.savefig(prob_space_plot_path, dpi=300, bbox_inches="tight")
plt.show()

metric_plot_df = spatial_summary_df.copy()
x_labels = metric_plot_df["stage_display"].tolist()
x = np.arange(len(x_labels))

for metric_name, ylabel, title, filename in [
    ("silhouette", "Silhouette Score", "Silhouette Score Across Training Stages", "silhouette_across_training_stages.png"),
    ("davies_bouldin", "Davies-Bouldin Index", "Davies-Bouldin Across Training Stages", "davies_bouldin_across_training_stages.png"),
    ("calinski_harabasz", "Calinski-Harabasz Index", "Calinski-Harabasz Across Training Stages", "calinski_harabasz_across_training_stages.png"),
    ("separation_ratio", "Centroid distance / intra-class spread", "Class Separation Ratio Across Training Stages", "separation_ratio_across_training_stages.png")
]:
    plt.figure(figsize=(8, 5))
    plt.plot(x, metric_plot_df[metric_name], marker="o")
    plt.xticks(x, x_labels, rotation=30, ha="right")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)

    save_path = os.path.join(SPATIAL_PLOTS_DIR, filename)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

print("\nDONE - Training + spatial analysis completed.")
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Training history:", history_path)
print("Test summary:", test_summary_path)
print("Test predictions:", test_predictions_path)
print("Spatial summary:", spatial_summary_path)
print("Changes table:", interpretation_path)
print("Compact thesis table:", compact_table_path)
print("Feature-space plot:", multi_stage_plot_path)
print("Probability-space plot:", prob_space_plot_path)

In [ ]:
# ============================================================
# CONTINUE AFTER CONFUSION MATRIX
# SPATIAL FEATURE-SPACE ANALYSIS ONLY
#
# Fixes PyTorch 2.6 torch.load issue using weights_only=False
#
# Requires from previous run:
# OUTPUT_DIR, CHECKPOINT_DIR, SPATIAL_CKPT_DIR, RESULTS_DIR
# SPATIAL_DIR, SPATIAL_PLOTS_DIR, SPATIAL_TABLES_DIR
# MODEL_NAME, IMG_SIZE, device, create_model, test_loader
# binary_name_map, SEED, MAX_TSNE_PER_CLASS
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from tqdm import tqdm

from sklearn.metrics import (
    davies_bouldin_score,
    calinski_harabasz_score,
    silhouette_score
)

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ============================================================
# 1. SAFETY: REDEFINE OUTPUT FOLDERS IF NEEDED
# ============================================================

SPATIAL_DIR = os.path.join(OUTPUT_DIR, "spatial_feature_space_analysis")
SPATIAL_PLOTS_DIR = os.path.join(SPATIAL_DIR, "plots")
SPATIAL_TABLES_DIR = os.path.join(SPATIAL_DIR, "tables")

for d in [SPATIAL_DIR, SPATIAL_PLOTS_DIR, SPATIAL_TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

# Αν δεν υπάρχει στο notebook:
try:
    binary_name_map
except NameError:
    binary_name_map = {
        0: "benign_like",
        1: "suspicious"
    }

try:
    MAX_TSNE_PER_CLASS
except NameError:
    MAX_TSNE_PER_CLASS = None

print("Spatial output:", SPATIAL_DIR)

# ============================================================
# 2. STAGE CHECKPOINTS
# ============================================================

stage_ckpts = {
    "before_training": os.path.join(SPATIAL_CKPT_DIR, "stage_before_training.pth"),
    "epoch_1": os.path.join(SPATIAL_CKPT_DIR, "stage_epoch_1.pth"),
    "epoch_5": os.path.join(SPATIAL_CKPT_DIR, "stage_epoch_5.pth"),
    "epoch_10": os.path.join(SPATIAL_CKPT_DIR, "stage_epoch_10.pth"),
    "best_model": os.path.join(SPATIAL_CKPT_DIR, "stage_best_model.pth")
}

for stage_name, ckpt_path in stage_ckpts.items():
    print(stage_name, "exists:", os.path.exists(ckpt_path), "|", ckpt_path)

missing = [k for k, v in stage_ckpts.items() if not os.path.exists(v)]

if len(missing) > 0:
    raise FileNotFoundError(
        "Λείπουν stage checkpoints: " + ", ".join(missing)
    )

# ============================================================
# 3. MODEL LOADER - FIXED FOR PYTORCH 2.6
# ============================================================

def load_stage_model(ckpt_path):
    ckpt = torch.load(
        ckpt_path,
        map_location=device,
        weights_only=False
    )

    m = create_model(pretrained=False).to(device)
    m.load_state_dict(ckpt["model_state"])
    m.eval()

    return m, ckpt

# ============================================================
# 4. FEATURE EXTRACTION
# ============================================================

@torch.no_grad()
def extract_features_for_spatial(m, loader):
    m.eval()

    feats = []
    labs = []
    probs = []

    for x, y in tqdm(loader, desc="Extracting features", leave=False):
        x = x.to(device)

        z = m.forward_features(x)

        try:
            z = m.forward_head(z, pre_logits=True)
        except Exception:
            if z.ndim == 4:
                z = z.mean(dim=(2, 3))
            elif z.ndim == 3:
                z = z.mean(dim=1)

        if z.ndim == 4:
            z = z.mean(dim=(2, 3))
        elif z.ndim == 3:
            z = z.mean(dim=1)

        logits = m(x).view(-1)
        p = torch.sigmoid(logits)

        feats.append(z.detach().cpu().numpy())
        labs.append(y.numpy().astype(int))
        probs.append(p.detach().cpu().numpy())

    feats = np.concatenate(feats)
    labs = np.concatenate(labs)
    probs = np.concatenate(probs)

    return feats, labs, probs

# ============================================================
# 5. BALANCED INDICES
# ============================================================

def make_balanced_indices(labels, max_per_class=None, seed=42):
    rng = np.random.default_rng(seed)
    labels = np.array(labels)
    classes = np.unique(labels)

    n_per_class = min([np.sum(labels == c) for c in classes])

    if max_per_class is not None:
        n_per_class = min(n_per_class, max_per_class)

    selected = []

    for c in classes:
        idx = np.where(labels == c)[0]
        chosen = rng.choice(idx, size=n_per_class, replace=False)
        selected.extend(chosen.tolist())

    selected = np.array(selected)
    rng.shuffle(selected)

    return selected

# Use best model once to get labels and create common balanced subset
tmp_model, _ = load_stage_model(stage_ckpts["best_model"])
tmp_features, tmp_labels, tmp_probs = extract_features_for_spatial(tmp_model, test_loader)

balanced_indices = make_balanced_indices(
    tmp_labels,
    max_per_class=MAX_TSNE_PER_CLASS,
    seed=SEED
)

del tmp_model
torch.cuda.empty_cache()

print("\nBalanced spatial subset:")
print("Total:", len(balanced_indices))
print("benign_like:", int(np.sum(tmp_labels[balanced_indices] == 0)))
print("suspicious:", int(np.sum(tmp_labels[balanced_indices] == 1)))

# ============================================================
# 6. SPATIAL METRICS
# ============================================================

def compute_spatial_metrics(features_2d, labels):
    labels = np.array(labels)

    db = davies_bouldin_score(features_2d, labels)
    ch = calinski_harabasz_score(features_2d, labels)
    sil = silhouette_score(features_2d, labels)

    class0 = features_2d[labels == 0]
    class1 = features_2d[labels == 1]

    centroid0 = class0.mean(axis=0)
    centroid1 = class1.mean(axis=0)

    centroid_distance = np.linalg.norm(centroid1 - centroid0)

    spread0 = np.mean(np.linalg.norm(class0 - centroid0, axis=1))
    spread1 = np.mean(np.linalg.norm(class1 - centroid1, axis=1))

    mean_intra_class_spread = (spread0 + spread1) / 2
    separation_ratio = centroid_distance / (mean_intra_class_spread + 1e-8)

    return {
        "davies_bouldin": db,
        "calinski_harabasz": ch,
        "silhouette": sil,
        "centroid_distance": centroid_distance,
        "benign_like_spread": spread0,
        "suspicious_spread": spread1,
        "mean_intra_class_spread": mean_intra_class_spread,
        "separation_ratio": separation_ratio
    }

# ============================================================
# 7. RUN PCA + t-SNE FOR ALL STAGES
# ============================================================

stage_order = [
    "before_training",
    "epoch_1",
    "epoch_5",
    "epoch_10",
    "best_model"
]

stage_display_names = {
    "before_training": "Before training",
    "epoch_1": "Epoch 1",
    "epoch_5": "Epoch 5",
    "epoch_10": "Epoch 10",
    "best_model": "Best model"
}

stage_results = []
stage_coordinates = {}

for stage_name in stage_order:
    print("\n" + "=" * 90)
    print("Spatial analysis stage:", stage_name)
    print("=" * 90)

    stage_model, stage_ckpt = load_stage_model(stage_ckpts[stage_name])

    features, labels, probs = extract_features_for_spatial(stage_model, test_loader)

    features_used = features[balanced_indices]
    labels_used = labels[balanced_indices]
    probs_used = probs[balanced_indices]

    scaler_features = StandardScaler()
    features_scaled = scaler_features.fit_transform(features_used)

    n_components = min(
        50,
        features_scaled.shape[1],
        features_scaled.shape[0] - 1
    )

    pca = PCA(
        n_components=n_components,
        random_state=SEED
    )

    features_pca = pca.fit_transform(features_scaled)
    pca_var = pca.explained_variance_ratio_.sum()

    perplexity = min(
        30,
        max(5, (features_pca.shape[0] - 1) // 3)
    )

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        learning_rate="auto",
        init="pca",
        max_iter=1500,
        random_state=SEED
    )

    features_2d = tsne.fit_transform(features_pca)

    metrics = compute_spatial_metrics(features_2d, labels_used)

    row = {
        "stage": stage_name,
        "stage_display": stage_display_names[stage_name],
        "epoch": stage_ckpt.get("epoch", 0),
        "num_samples": len(labels_used),
        "num_benign_like": int(np.sum(labels_used == 0)),
        "num_suspicious": int(np.sum(labels_used == 1)),
        "pca_components": n_components,
        "pca_explained_variance": pca_var,
        "tsne_perplexity": perplexity,
        **metrics
    }

    stage_results.append(row)

    coords_df = pd.DataFrame({
        "stage": stage_name,
        "stage_display": stage_display_names[stage_name],
        "tsne_1": features_2d[:, 0],
        "tsne_2": features_2d[:, 1],
        "label": labels_used,
        "class_name": [binary_name_map[int(x)] for x in labels_used],
        "prob_suspicious": probs_used
    })

    stage_coordinates[stage_name] = coords_df

    coord_path = os.path.join(
        SPATIAL_TABLES_DIR,
        f"coordinates_{stage_name}.csv"
    )

    coords_df.to_csv(coord_path, index=False)

    print("Saved coordinates:", coord_path)
    print("Metrics:", row)

    del stage_model
    torch.cuda.empty_cache()

# ============================================================
# 8. COMPARATIVE TABLES
# ============================================================

spatial_summary_df = pd.DataFrame(stage_results)

spatial_summary_path = os.path.join(
    SPATIAL_TABLES_DIR,
    "spatial_stage_comparison_table.csv"
)

spatial_summary_df.to_csv(spatial_summary_path, index=False)

print("\n=== SPATIAL STAGE COMPARISON TABLE ===")
display(spatial_summary_df)

baseline_row = spatial_summary_df[
    spatial_summary_df["stage"] == "before_training"
].iloc[0]

interpret_rows = []

for _, row in spatial_summary_df.iterrows():
    interpret_rows.append({
        "stage": row["stage_display"],
        "davies_bouldin": row["davies_bouldin"],
        "davies_bouldin_change_vs_before": row["davies_bouldin"] - baseline_row["davies_bouldin"],
        "calinski_harabasz": row["calinski_harabasz"],
        "calinski_change_vs_before": row["calinski_harabasz"] - baseline_row["calinski_harabasz"],
        "silhouette": row["silhouette"],
        "silhouette_change_vs_before": row["silhouette"] - baseline_row["silhouette"],
        "centroid_distance": row["centroid_distance"],
        "centroid_distance_change_vs_before": row["centroid_distance"] - baseline_row["centroid_distance"],
        "mean_intra_class_spread": row["mean_intra_class_spread"],
        "separation_ratio": row["separation_ratio"],
        "separation_ratio_change_vs_before": row["separation_ratio"] - baseline_row["separation_ratio"]
    })

interpretation_df = pd.DataFrame(interpret_rows)

interpretation_path = os.path.join(
    SPATIAL_TABLES_DIR,
    "spatial_stage_changes_vs_before_training.csv"
)

interpretation_df.to_csv(interpretation_path, index=False)

print("\n=== WHAT CHANGED COMPARED TO BEFORE TRAINING ===")
display(interpretation_df)

compact_table_df = spatial_summary_df[[
    "stage_display",
    "epoch",
    "davies_bouldin",
    "calinski_harabasz",
    "silhouette",
    "centroid_distance",
    "mean_intra_class_spread",
    "separation_ratio"
]].copy()

compact_table_df = compact_table_df.rename(columns={
    "stage_display": "Stage",
    "epoch": "Epoch",
    "davies_bouldin": "Davies-Bouldin ↓",
    "calinski_harabasz": "Calinski-Harabasz ↑",
    "silhouette": "Silhouette ↑",
    "centroid_distance": "Centroid distance ↑",
    "mean_intra_class_spread": "Mean intra-class spread ↓",
    "separation_ratio": "Separation ratio ↑"
})

compact_table_path = os.path.join(
    SPATIAL_TABLES_DIR,
    "compact_spatial_comparison_table_for_thesis.csv"
)

compact_table_df.to_csv(compact_table_path, index=False)

print("\n=== COMPACT THESIS TABLE ===")
display(compact_table_df)

# ============================================================
# 9. PLOTS
# ============================================================

fig, axes = plt.subplots(
    1,
    len(stage_order),
    figsize=(5 * len(stage_order), 5)
)

for ax, stage_name in zip(axes, stage_order):
    coords = stage_coordinates[stage_name]

    for label_value, class_name in binary_name_map.items():
        mask = coords["label"].values == label_value

        ax.scatter(
            coords.loc[mask, "tsne_1"],
            coords.loc[mask, "tsne_2"],
            s=24,
            alpha=0.82,
            label=class_name
        )

    row = spatial_summary_df[
        spatial_summary_df["stage"] == stage_name
    ].iloc[0]

    ax.set_title(
        f"{stage_display_names[stage_name]}\n"
        f"Sil={row['silhouette']:.3f} | DB={row['davies_bouldin']:.3f}"
    )

    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.grid(True)

axes[0].legend()

plt.suptitle(
    "Feature Space Evolution Across Training Stages\n"
    "ISIC 2019 binary suspicious vs benign-like",
    fontsize=16
)

plt.tight_layout()

multi_stage_plot_path = os.path.join(
    SPATIAL_PLOTS_DIR,
    "feature_space_evolution_by_class.png"
)

plt.savefig(multi_stage_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", multi_stage_plot_path)

fig, axes = plt.subplots(
    1,
    len(stage_order),
    figsize=(5 * len(stage_order), 5)
)

for ax, stage_name in zip(axes, stage_order):
    coords = stage_coordinates[stage_name]

    sc = ax.scatter(
        coords["tsne_1"],
        coords["tsne_2"],
        c=coords["prob_suspicious"],
        s=24,
        alpha=0.85,
        vmin=0,
        vmax=1
    )

    row = spatial_summary_df[
        spatial_summary_df["stage"] == stage_name
    ].iloc[0]

    ax.set_title(
        f"{stage_display_names[stage_name]}\n"
        f"CH={row['calinski_harabasz']:.1f}"
    )

    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.grid(True)

fig.colorbar(
    sc,
    ax=axes.ravel().tolist(),
    shrink=0.75,
    label="Probability suspicious"
)

plt.suptitle(
    "Decision Confidence Evolution in Feature Space\n"
    "Color indicates probability for suspicious class",
    fontsize=16
)

prob_space_plot_path = os.path.join(
    SPATIAL_PLOTS_DIR,
    "feature_space_evolution_by_probability.png"
)

plt.savefig(prob_space_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", prob_space_plot_path)

metric_plot_df = spatial_summary_df.copy()
x_labels = metric_plot_df["stage_display"].tolist()
x = np.arange(len(x_labels))

metric_specs = [
    ("silhouette", "Silhouette Score", "Silhouette Score Across Training Stages", "silhouette_across_training_stages.png"),
    ("davies_bouldin", "Davies-Bouldin Index", "Davies-Bouldin Across Training Stages", "davies_bouldin_across_training_stages.png"),
    ("calinski_harabasz", "Calinski-Harabasz Index", "Calinski-Harabasz Across Training Stages", "calinski_harabasz_across_training_stages.png"),
    ("separation_ratio", "Centroid distance / intra-class spread", "Class Separation Ratio Across Training Stages", "separation_ratio_across_training_stages.png")
]

for metric_name, ylabel, title, filename in metric_specs:
    plt.figure(figsize=(8, 5))
    plt.plot(x, metric_plot_df[metric_name], marker="o")
    plt.xticks(x, x_labels, rotation=30, ha="right")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)

    save_path = os.path.join(SPATIAL_PLOTS_DIR, filename)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)

print("\nSpatial analysis completed.")
print("Spatial summary:", spatial_summary_path)
print("Changes table:", interpretation_path)
print("Compact thesis table:", compact_table_path)
print("Feature-space plot:", multi_stage_plot_path)
print("Probability-space plot:", prob_space_plot_path)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Plot distribution of probabilities by true label
plt.figure(figsize=(10, 6))
sns.histplot(
    data=test_predictions_df,
    x='prob_suspicious',
    hue='true_class',
    kde=True,
    palette={'benign_like': 'blue', 'suspicious': 'red'},
    alpha=0.6
)
plt.title('Distribution of Probability Suspicious by True Class')
plt.xlabel('Probability of Suspicious Class')
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()


In [ ]:
# Plot distribution of probabilities for correct vs. incorrect predictions
plt.figure(figsize=(10, 6))
sns.histplot(
    data=test_predictions_df,
    x='prob_suspicious',
    hue='correct',
    kde=True,
    palette={True: 'green', False: 'orange'},
    alpha=0.6
)
plt.title('Distribution of Probability Suspicious by Prediction Correctness')
plt.xlabel('Probability of Suspicious Class')
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()


In [ ]:
!pip install -q grad-cam captum

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from PIL import Image
import torch
import torch.nn as nn
import torchvision.transforms as T

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

from captum.attr import GradientShap

import timm

# ============================================================
# 1. OUTPUT FOLDERS
# ============================================================

XAI_DIR = os.path.join(OUTPUT_DIR, "xai_outputs")
GRADCAM_DIR = os.path.join(XAI_DIR, "gradcam")
GRADCAMPP_DIR = os.path.join(XAI_DIR, "gradcampp")
ACROSS_DIR = os.path.join(XAI_DIR, "across_epochs_gradcam_gradcampp")
SHAP_DIR = os.path.join(XAI_DIR, "shap_block_size_20")

for d in [XAI_DIR, GRADCAM_DIR, GRADCAMPP_DIR, ACROSS_DIR, SHAP_DIR]:
    os.makedirs(d, exist_ok=True)

print("XAI_DIR:", XAI_DIR)

# ============================================================
# ADDED FROM CYL255JLL45Q: MODEL & IMAGE LOADING FUNCTIONS
# ============================================================

# Assuming MODEL_NAME, IMG_SIZE, and device are defined in a previous cell and accessible.
# Also assuming eval_tf is defined in a previous cell and accessible.

def create_model(pretrained=True):
    model = timm.create_model(
        MODEL_NAME,
        pretrained=pretrained,
        num_classes=1
    )
    return model

def safe_open_rgb(path):
    try:
        img = cv2.imread(path, cv2.IMREAD_COLOR)

        if img is None:
            raise ValueError("cv2.imread returned None")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return Image.fromarray(img)

    except Exception as e:
        print("Image error:", path, e)
        arr = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        return Image.fromarray(arr)

# Also added eval_tf definition (assuming it's needed)
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

# Assuming `device` and `MODEL_NAME` are already defined in the global scope from the training notebook.
# If not, they would need to be added here as well, or the user would need to ensure the training cell is run first.

# ============================================================
# 2. LOAD BEST MODEL AND PREDICTIONS
# ============================================================

BEST_CKPT_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pth")
TEST_PRED_PATH = os.path.join(RESULTS_DIR, "test_predictions.csv")

print("Best checkpoint exists:", os.path.exists(BEST_CKPT_PATH))
print("Test predictions exists:", os.path.exists(TEST_PRED_PATH))

test_results_df = pd.read_csv(TEST_PRED_PATH)

def load_model_from_checkpoint(ckpt_path):
    # Explicitly set weights_only=False to address UnpicklingError in PyTorch 2.6+
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    m = create_model(pretrained=False).to(device)
    m.load_state_dict(ckpt["model_state"])
    m.eval()

    return m, ckpt

best_model, best_ckpt = load_model_from_checkpoint(BEST_CKPT_PATH)

print("Loaded best model.")
print("Best epoch:", best_ckpt.get("epoch"))
print("Threshold:", best_ckpt.get("threshold"))

label_name_map = {
    0: "benign_like",
    1: "suspicious"
}

# ============================================================
# 3. SELECT CASES
# ============================================================

def select_xai_cases(pred_df):
    dfp = pred_df.copy()

    if "correct" not in dfp.columns:
        dfp["correct"] = dfp["true_label"] == dfp["pred_label"]

    cases = {}

    correct_suspicious = dfp[
        (dfp["true_label"] == 1) &
        (dfp["pred_label"] == 1)
    ].copy()

    if len(correct_suspicious) > 0:
        correct_suspicious["score"] = np.abs(correct_suspicious["prob_suspicious"] - 0.90)
        cases["correct_suspicious"] = correct_suspicious.sort_values("score").index[0]

    correct_benign = dfp[
        (dfp["true_label"] == 0) &
        (dfp["pred_label"] == 0)
    ].copy()

    if len(correct_benign) > 0:
        correct_benign["score"] = np.abs(correct_benign["prob_suspicious"] - 0.20)
        cases["correct_benign_like"] = correct_benign.sort_values("score").index[0]

    wrong = dfp[dfp["correct"] == False].copy()

    if len(wrong) > 0:
        wrong["wrong_confidence"] = np.where(
            wrong["pred_label"] == 1,
            wrong["prob_suspicious"],
            1.0 - wrong["prob_suspicious"]
        )

        cases["wrong_prediction"] = wrong.sort_values(
            "wrong_confidence",
            ascending=False
        ).index[0]

    return cases

xai_cases = select_xai_cases(test_results_df)

print("Selected XAI cases:")
print(xai_cases)

# ============================================================
# 4. HELPERS
# ============================================================

class BinaryTwoOutputWrapper(nn.Module):
    def __init__(self, binary_model):
        super().__init__()
        self.binary_model = binary_model

    def forward(self, x):
        logit = self.binary_model(x).view(-1)
        return torch.stack([-logit, logit], dim=1)

def get_target_layer_efficientnet(m):
    if hasattr(m, "blocks"):
        return m.blocks[-1]
    if hasattr(m, "features"):
        return m.features[-1]
    raise AttributeError("Δεν βρέθηκε target layer.")

def load_rgb_np_and_tensor(img_path):
    pil_img = safe_open_rgb(img_path)
    pil_show = pil_img.resize((IMG_SIZE, IMG_SIZE))

    rgb_np = np.array(pil_show).astype(np.float32) / 255.0
    x = eval_tf(pil_img).unsqueeze(0).to(device)

    return pil_show, rgb_np, x

# ============================================================
# 5. GRAD-CAM AND GRAD-CAM++
# ============================================================

def plot_gradcam_pair(idx, case_name):
    row = test_results_df.loc[idx]

    img_path = row["image_path"]

    true_label = int(row["true_label"])
    pred_label = int(row["pred_label"])
    prob = float(row["prob_suspicious"])
    correct = bool(row["correct"])

    true_name = label_name_map[true_label]
    pred_name = label_name_map[pred_label]

    pil_img, rgb_np, x = load_rgb_np_and_tensor(img_path)

    cam_model = BinaryTwoOutputWrapper(best_model).to(device).eval()
    target_layer = get_target_layer_efficientnet(best_model)

    cam_methods = {
        "Grad-CAM": GradCAM,
        "Grad-CAM++": GradCAMPlusPlus
    }

    results = {}

    for method_name, method_class in cam_methods.items():
        cam = method_class(
            model=cam_model,
            target_layers=[target_layer]
        )

        grayscale_cam = cam(
            input_tensor=x,
            targets=[ClassifierOutputTarget(pred_label)]
        )[0]

        overlay = show_cam_on_image(
            rgb_np,
            grayscale_cam,
            use_rgb=True,
            image_weight=0.78
        )

        results[method_name] = {
            "heatmap": grayscale_cam,
            "overlay": overlay
        }

    plt.figure(figsize=(20, 5))

    plt.subplot(1, 5, 1)
    plt.imshow(pil_img)
    plt.axis("off")
    plt.title(
        f"Original\n"
        f"True={true_name}\n"
        f"Pred={pred_name}\n"
        f"Prob={prob:.3f}"
    )

    plt.subplot(1, 5, 2)
    plt.imshow(results["Grad-CAM"]["heatmap"], cmap="jet")
    plt.axis("off")
    plt.title("Grad-CAM")

    plt.subplot(1, 5, 3)
    plt.imshow(results["Grad-CAM"]["overlay"])
    plt.axis("off")
    plt.title("Grad-CAM overlay")

    plt.subplot(1, 5, 4)
    plt.imshow(results["Grad-CAM++"]["heatmap"], cmap="jet")
    plt.axis("off")
    plt.title("Grad-CAM++")

    plt.subplot(1, 5, 5)
    plt.imshow(results["Grad-CAM++"]["overlay"])
    plt.axis("off")
    plt.title("Grad-CAM++ overlay")

    plt.suptitle(
        f"{case_name} | Correct={correct}",
        fontsize=14
    )

    plt.tight_layout()

    save_path = os.path.join(
        XAI_DIR,
        f"{case_name}_gradcam_gradcampp.png"
    )

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)

for case_name, idx in xai_cases.items():
    plot_gradcam_pair(idx, case_name)

# ============================================================
# 6. ACROSS EPOCHS 1, 5, 10, BEST
# ============================================================

def compute_cam_overlay_from_checkpoint(ckpt_path, img_path, target_class, method="gradcampp"):
    temp_model, _ = load_model_from_checkpoint(ckpt_path)

    cam_model = BinaryTwoOutputWrapper(temp_model).to(device).eval()
    target_layer = get_target_layer_efficientnet(temp_model)

    pil_img, rgb_np, x = load_rgb_np_and_tensor(img_path)

    if method == "gradcam":
        cam_class = GradCAM
    else:
        cam_class = GradCAMPlusPlus

    cam = cam_class(
        model=cam_model,
        target_layers=[target_layer]
    )

    grayscale_cam = cam(
        input_tensor=x,
        targets=[ClassifierOutputTarget(target_class)]
    )[0]

    overlay = show_cam_on_image(
        rgb_np,
        grayscale_cam,
        use_rgb=True,
        image_weight=0.78
    )

    del temp_model
    torch.cuda.empty_cache()

    return overlay

def plot_across_epochs(idx, case_name, method="gradcampp"):
    row = test_results_df.loc[idx]

    img_path = row["image_path"]

    true_label = int(row["true_label"])
    pred_label = int(row["pred_label"])
    prob = float(row["prob_suspicious"])

    true_name = label_name_map[true_label]
    pred_name = label_name_map[pred_label]

    target_class = true_label

    pil_img = safe_open_rgb(img_path).resize((IMG_SIZE, IMG_SIZE))

    plot_items = []

    for ep in [1, 5, 10]:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"epoch_{ep}.pth")

        if os.path.exists(ckpt_path):
            plot_items.append((f"Epoch {ep}", ckpt_path))
        else:
            print("Missing:", ckpt_path)

    plot_items.append((f"Best epoch {best_ckpt.get('epoch')}", BEST_CKPT_PATH))

    n_cols = len(plot_items) + 1

    plt.figure(figsize=(4.3 * n_cols, 4.8))

    plt.subplot(1, n_cols, 1)
    plt.imshow(pil_img)
    plt.axis("off")
    plt.title(
        f"Original\n"
        f"True={true_name}\n"
        f"Pred={pred_name}\n"
        f"Prob={prob:.3f}"
    )

    for col_idx, (title, ckpt_path) in enumerate(plot_items, start=2):
        overlay = compute_cam_overlay_from_checkpoint(
            ckpt_path=ckpt_path,
            img_path=img_path,
            target_class=target_class,
            method=method
        )

        plt.subplot(1, n_cols, col_idx)
        plt.imshow(overlay)
        plt.axis("off")
        plt.title(title)

    method_title = "Grad-CAM++" if method == "gradcampp" else "Grad-CAM"

    plt.suptitle(
        f"{method_title} Across Epochs - {case_name}\n"
        f"Target class: {true_name}",
        fontsize=14
    )

    plt.tight_layout()

    save_path = os.path.join(
        ACROSS_DIR,
        f"{case_name}_{method}_across_epochs.png"
    )

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)

for case_name, idx in xai_cases.items():
    plot_across_epochs(idx, case_name, method="gradcam")
    plot_across_epochs(idx, case_name, method="gradcampp")

# ============================================================
# 7. BLOCK-STYLE GRADIENT SHAP, SMALL PIXELS
# ============================================================

BLOCK_SIZE = 20
N_SAMPLES = 40
STDEVS = 0.06

wrapped_model = BinaryTwoOutputWrapper(best_model).to(device).eval()

def compute_gradient_shap_block(input_tensor, target_class, block_size=20):
    gradient_shap = GradientShap(wrapped_model)

    baseline_black = torch.zeros_like(input_tensor).to(device)
    baseline_gray = torch.zeros_like(input_tensor).to(device) + 0.5
    baseline_noise = torch.randn_like(input_tensor).to(device) * 0.05

    baselines = torch.cat(
        [baseline_black, baseline_gray, baseline_noise],
        dim=0
    )

    attr = gradient_shap.attribute(
        input_tensor,
        baselines=baselines,
        target=target_class,
        n_samples=N_SAMPLES,
        stdevs=STDEVS
    )

    attr = attr.detach().cpu()[0]

    signed_map = attr.mean(dim=0).numpy()
    signed_map = cv2.GaussianBlur(signed_map, (5, 5), 0)

    block_map = cv2.resize(
        signed_map,
        (block_size, block_size),
        interpolation=cv2.INTER_AREA
    )

    return block_map

def plot_shap_block(idx, case_name, explain_class="suspicious"):
    row = test_results_df.loc[idx]

    img_path = row["image_path"]

    true_label = int(row["true_label"])
    pred_label = int(row["pred_label"])
    prob = float(row["prob_suspicious"])

    true_name = label_name_map[true_label]
    pred_name = label_name_map[pred_label]

    if explain_class == "pred":
        target_class = pred_label
    elif explain_class == "true":
        target_class = true_label
    else:
        target_class = 1

    target_name = label_name_map[target_class]

    pil_img = safe_open_rgb(img_path).resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(pil_img).astype(np.float32) / 255.0

    input_tensor = eval_tf(pil_img).unsqueeze(0).to(device)

    block_map = compute_gradient_shap_block(
        input_tensor=input_tensor,
        target_class=target_class,
        block_size=BLOCK_SIZE
    )

    vmax = np.max(np.abs(block_map)) + 1e-12
    vmin = -vmax

    fig = plt.figure(figsize=(12, 5))

    fig.suptitle(
        f"SHAP - explaining {target_name}\n"
        f"True={true_name} | Pred={pred_name} | Prob suspicious={prob:.3f}",
        fontsize=13
    )

    ax1 = plt.subplot(1, 2, 1)
    ax1.imshow(img_np)
    ax1.set_title("Original", fontsize=12)
    ax1.axis("off")

    ax2 = plt.subplot(1, 2, 2)
    im = ax2.imshow(
        block_map,
        cmap="bwr",
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest"
    )
    ax2.set_title("SHAP values", fontsize=12)
    ax2.axis("off")

    cbar_ax = fig.add_axes([0.23, 0.08, 0.54, 0.055])
    cbar = fig.colorbar(im, cax=cbar_ax, orientation="horizontal")
    cbar.set_label("SHAP value", fontsize=10)

    plt.subplots_adjust(top=0.78, bottom=0.20, wspace=0.10)

    save_path = os.path.join(
        SHAP_DIR,
        f"{case_name}_shap_block_size_{BLOCK_SIZE}.png"
    )

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)

for case_name, idx in xai_cases.items():
    plot_shap_block(
        idx=idx,
        case_name=case_name,
        explain_class="suspicious"
    )

print("\nDONE - Explainability completed.")
print("Grad-CAM / Grad-CAM++:", XAI_DIR)
print("Across epochs:", ACROSS_DIR)
print("SHAP:", SHAP_DIR)


In [ ]:
# ============================================================
# CLEAR GPU MEMORY
# ============================================================

import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared.")

In [ ]:
# ============================================================
# LIGHTWEIGHT INTEGRATED GRADIENTS + SMOOTHGRAD
# Low-memory version
# ============================================================

!pip install -q captum

import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn

from captum.attr import IntegratedGradients, NoiseTunnel

# ============================================================
# 1. CLEAR MEMORY
# ============================================================

gc.collect()
torch.cuda.empty_cache()

# ============================================================
# 2. PATHS
# ============================================================

IG_DIR = os.path.join(OUTPUT_DIR, "xai_outputs", "integrated_gradients_light")
os.makedirs(IG_DIR, exist_ok=True)

BEST_CKPT_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pth")
TEST_PRED_PATH = os.path.join(RESULTS_DIR, "test_predictions.csv")

print("BEST_CKPT_PATH exists:", os.path.exists(BEST_CKPT_PATH))
print("TEST_PRED_PATH exists:", os.path.exists(TEST_PRED_PATH))
print("IG_DIR:", IG_DIR)

# ============================================================
# 3. LOAD MODEL
# ============================================================

def load_model_from_checkpoint(ckpt_path):
    ckpt = torch.load(
        ckpt_path,
        map_location=device,
        weights_only=False
    )

    m = create_model(pretrained=False).to(device)
    m.load_state_dict(ckpt["model_state"])
    m.eval()

    return m, ckpt

best_model, best_ckpt = load_model_from_checkpoint(BEST_CKPT_PATH)
best_model.eval()

print("Loaded best model.")
print("Best epoch:", best_ckpt.get("epoch"))
print("Threshold:", best_ckpt.get("threshold"))

# ============================================================
# 4. LOAD PREDICTIONS
# ============================================================

test_results_df = pd.read_csv(TEST_PRED_PATH)

label_name_map = {
    0: "benign_like",
    1: "suspicious"
}

# ============================================================
# 5. SELECT ONLY 2 CASES TO SAVE MEMORY
# ============================================================

def select_ig_cases_light(pred_df):
    dfp = pred_df.copy()

    if "correct" not in dfp.columns:
        dfp["correct"] = dfp["true_label"] == dfp["pred_label"]

    cases = {}

    # σωστό suspicious, αλλά όχι 1.000 αν γίνεται
    correct_suspicious = dfp[
        (dfp["true_label"] == 1) &
        (dfp["pred_label"] == 1)
    ].copy()

    if len(correct_suspicious) > 0:
        correct_suspicious["score"] = np.abs(correct_suspicious["prob_suspicious"] - 0.85)
        cases["correct_suspicious"] = correct_suspicious.sort_values("score").index[0]

    # σωστό benign-like
    correct_benign = dfp[
        (dfp["true_label"] == 0) &
        (dfp["pred_label"] == 0)
    ].copy()

    if len(correct_benign) > 0:
        correct_benign["score"] = np.abs(correct_benign["prob_suspicious"] - 0.20)
        cases["correct_benign_like"] = correct_benign.sort_values("score").index[0]

    return cases

ig_cases = select_ig_cases_light(test_results_df)

print("Selected IG cases:")
print(ig_cases)

# ============================================================
# 6. MODEL WRAPPER
# ============================================================

class BinaryTwoOutputWrapper(nn.Module):
    def __init__(self, binary_model):
        super().__init__()
        self.binary_model = binary_model

    def forward(self, x):
        logit = self.binary_model(x).view(-1)
        return torch.stack([-logit, logit], dim=1)

wrapped_model = BinaryTwoOutputWrapper(best_model).to(device).eval()

# ============================================================
# 7. HELPERS
# ============================================================

def load_rgb_np_and_tensor(img_path):
    pil_img = safe_open_rgb(img_path)
    pil_show = pil_img.resize((IMG_SIZE, IMG_SIZE))

    rgb_np = np.array(pil_show).astype(np.float32) / 255.0
    x = eval_tf(pil_img).unsqueeze(0).to(device)

    return pil_show, rgb_np, x

def normalize_map(x):
    x = x.astype(np.float32)
    x = x - x.min()
    if x.max() > 0:
        x = x / x.max()
    return x

def make_overlay(rgb_np, heatmap_2d, alpha=0.40):
    heatmap_2d = normalize_map(heatmap_2d)

    heatmap_uint8 = np.uint8(255 * heatmap_2d)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    overlay = (
        alpha * heatmap_color.astype(np.float32) / 255.0
        + (1 - alpha) * rgb_np
    )

    overlay = np.clip(overlay, 0, 1)
    return overlay

# ============================================================
# 8. LOW-MEMORY IG + SMOOTHGRAD
# ============================================================

ig = IntegratedGradients(wrapped_model)
nt = NoiseTunnel(ig)

def compute_ig_maps_light(input_tensor, target_class=1):
    baseline = torch.zeros_like(input_tensor).to(device)

    # απλό Integrated Gradients
    attr_ig = ig.attribute(
        input_tensor,
        baselines=baseline,
        target=target_class,
        n_steps=24,
        internal_batch_size=4
    )

    gc.collect()
    torch.cuda.empty_cache()

    # SmoothGrad πάνω στο IG, πολύ πιο ελαφρύ
    attr_sg = nt.attribute(
        input_tensor,
        baselines=baseline,
        target=target_class,
        nt_type="smoothgrad_sq",
        nt_samples=6,
        stdevs=0.05,
        n_steps=16,
        internal_batch_size=2
    )

    attr_ig = attr_ig.detach().cpu()[0].numpy()
    attr_sg = attr_sg.detach().cpu()[0].numpy()

    ig_signed = attr_ig.mean(axis=0)
    sg_signed = attr_sg.mean(axis=0)

    ig_abs = np.abs(attr_ig).mean(axis=0)
    sg_abs = np.abs(attr_sg).mean(axis=0)

    ig_signed = cv2.GaussianBlur(ig_signed, (5, 5), 0)
    sg_signed = cv2.GaussianBlur(sg_signed, (5, 5), 0)
    ig_abs = cv2.GaussianBlur(ig_abs, (5, 5), 0)
    sg_abs = cv2.GaussianBlur(sg_abs, (5, 5), 0)

    return {
        "ig_signed": ig_signed,
        "ig_abs": ig_abs,
        "sg_signed": sg_signed,
        "sg_abs": sg_abs
    }

# ============================================================
# 9. PLOT
# ============================================================

def plot_ig_explanation_light(idx, case_name, explain_class="suspicious"):
    gc.collect()
    torch.cuda.empty_cache()

    row = test_results_df.loc[idx]

    img_path = row["image_path"]

    true_label = int(row["true_label"])
    pred_label = int(row["pred_label"])
    prob = float(row["prob_suspicious"])
    correct = bool(row["correct"])

    true_name = label_name_map[true_label]
    pred_name = label_name_map[pred_label]

    if explain_class == "pred":
        target_class = pred_label
    elif explain_class == "true":
        target_class = true_label
    else:
        target_class = 1

    target_name = label_name_map[target_class]

    pil_img, rgb_np, input_tensor = load_rgb_np_and_tensor(img_path)

    maps = compute_ig_maps_light(
        input_tensor=input_tensor,
        target_class=target_class
    )

    ig_signed = maps["ig_signed"]
    ig_abs = maps["ig_abs"]
    sg_signed = maps["sg_signed"]
    sg_abs = maps["sg_abs"]

    ig_overlay = make_overlay(rgb_np, ig_abs, alpha=0.40)
    sg_overlay = make_overlay(rgb_np, sg_abs, alpha=0.40)

    vmax_ig = np.max(np.abs(ig_signed)) + 1e-12
    vmax_sg = np.max(np.abs(sg_signed)) + 1e-12

    fig = plt.figure(figsize=(18, 9))

    fig.suptitle(
        f"Integrated Gradients / SmoothGrad - explaining {target_name}\n"
        f"Case={case_name} | True={true_name} | Pred={pred_name} | "
        f"Prob suspicious={prob:.3f} | Correct={correct}",
        fontsize=14
    )

    ax1 = plt.subplot(2, 3, 1)
    ax1.imshow(rgb_np)
    ax1.set_title("Original")
    ax1.axis("off")

    ax2 = plt.subplot(2, 3, 2)
    im1 = ax2.imshow(ig_signed, cmap="bwr", vmin=-vmax_ig, vmax=vmax_ig)
    ax2.set_title("Integrated Gradients signed")
    ax2.axis("off")

    ax3 = plt.subplot(2, 3, 3)
    ax3.imshow(ig_overlay)
    ax3.set_title("IG overlay")
    ax3.axis("off")

    ax4 = plt.subplot(2, 3, 4)
    ax4.imshow(rgb_np)
    ax4.set_title("Original")
    ax4.axis("off")

    ax5 = plt.subplot(2, 3, 5)
    im2 = ax5.imshow(sg_signed, cmap="bwr", vmin=-vmax_sg, vmax=vmax_sg)
    ax5.set_title("SmoothGrad over IG signed")
    ax5.axis("off")

    ax6 = plt.subplot(2, 3, 6)
    ax6.imshow(sg_overlay)
    ax6.set_title("SmoothGrad overlay")
    ax6.axis("off")

    plt.tight_layout()

    save_path = os.path.join(
        IG_DIR,
        f"{case_name}_integrated_gradients_light.png"
    )

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)

    del input_tensor, maps
    gc.collect()
    torch.cuda.empty_cache()

# ============================================================
# 10. RUN
# ============================================================

for case_name, idx in ig_cases.items():
    plot_ig_explanation_light(
        idx=idx,
        case_name=case_name,
        explain_class="suspicious"
    )

print("\nIntegrated Gradients / SmoothGrad light completed.")
print("Saved in:", IG_DIR)